In [ ]:
# Euchre Calculator -- a worked run of the engine as it currently stands:
# deal a hand, solve the whole auction, play out the contract that survives.
#
# Cards are written the way you would say them: JS is the jack of spades, TH
# the ten of hearts. The solver underneath still works in the 2-D vector
# encoding with spades fixed as trump; rotation translates in and out, so
# nothing at this level has to think about it.
import collections
import random
import time

import numpy as np

import bidding as b
import game
import rotation as r
from fast_search import solve_line, _decode

In [ ]:
# 1. Deal a hand.
#
# game.Deal is the dealt state as a real game has it: four hands, the up-card
# turned off the kitty, three cards buried under it, and a dealer. Nobody has
# called anything yet, so there is no trump.
#
# The dealer bids LAST. With seat 3 dealing, the eldest hand -- first to speak
# -- is seat 0, and the order runs round to the dealer.
deal = game.deal_random(rng=random.Random(7), dealer=3)

print(deal.describe())
print("\nbidding order:", deal.bidding_order(), "(eldest first, dealer last)")

  seat 0: AC 9S KS QH TD
  seat 1: JH AS QS 9D TS
  seat 2: TH KH JD 9C QD
  seat 3 (dealer): AD QC AH JC TC
  up-card: JS
  buried:  9H KC KD

bidding order: [0, 1, 2, 3] (eldest first, dealer last)


In [ ]:
# 2. Solve the auction.
#
# Every seat sees every hand and bids to maximise its own team, knowing exactly
# how the rest of the auction and the play will go. Values are net points to
# team 0 (seats 0 and 2), so calls by different seats sit on one scale.
#
# Note who chooses the discard: the DEALER does, even when the opposition
# ordered it up. Here seat 3 both deals and calls, so it pitches to help
# itself -- but when the other team orders, the dealer pitches to hurt them.
outcome = b.solve_bidding(deal)

for step in outcome.line:
    print(" ", step)
print("\n->", outcome)

  seat 0 passes
  seat 1 passes
  seat 2 passes
  seat 3 orders up spades

-> seat 3 ordered up spades (dealer pitched AD) -> -2 to team 0


In [ ]:
# 3. Play out the contract.
#
# The auction hands back a Contract carrying the deal AFTER the pickup and
# discard. That is a solvable position, so the trick-play solver takes it from
# here. Reading the line back through rotation puts it in card names rather
# than raw vectors.
def show_play(contract):
    """Print one optimal line of play, in card names."""
    d = contract.deal
    hands = r.deal_to_engine(d.hands, contract.trump)
    score, ps, pv, pp, winners = solve_line(hands, d.first_bidder, contract.caller)

    print("trump %s, called by seat %d, led by seat %d\n"
          % (r.suit_name(contract.trump), contract.caller, d.first_bidder))
    for t in range(5):
        played = ["seat %d %s"
                  % (int(pp[t, k]),
                     r.card_name(r.card_from_engine(_decode(ps[t, k], pv[t, k]),
                                                    contract.trump)))
                  for k in range(4)]
        print("  trick %d: %-46s won by seat %d"
              % (t + 1, ", ".join(played), int(winners[t])))

    tricks = sum(1 for w in winners if w % 2 == contract.caller % 2)
    print("\n  calling team took %d tricks -> %+d" % (tricks, score))


show_play(outcome.contract)

trump spades, called by seat 3, led by seat 0

  trick 1: seat 0 AC, seat 1 AS, seat 2 9C, seat 3 QC     won by seat 1
  trick 2: seat 1 JH, seat 2 TH, seat 3 AH, seat 0 QH     won by seat 3
  trick 3: seat 3 JS, seat 0 9S, seat 1 TS, seat 2 QD     won by seat 3
  trick 4: seat 3 JC, seat 0 KS, seat 1 QS, seat 2 JD     won by seat 3
  trick 5: seat 3 TC, seat 0 TD, seat 1 9D, seat 2 KH     won by seat 3

  calling team took 5 tricks -> +2


In [ ]:
# 4. "Should I order this up?" -- the question the calculator exists to answer.
#
# Pin the hand you can actually see, yours plus the up-card, and let everything
# else be dealt. first_bid_choice returns the two options from the eldest
# hand's OWN point of view, so the bigger number is the better bid.
#
# Seat 3 deals, so seat 0 is eldest and speaks first. On this one layout,
# passing wins.
my_seat = 0
my_hand = r.parse_hand("JS AS 9H 9D TC")      # right bower, ace of trump, junk
up_card = r.parse_card("9S")

one = game.deal_around(known_hand=my_hand, seat=my_seat, up_card=up_card,
                       rng=random.Random(1), dealer=3)

ordered, passed = b.first_bid_choice(one)
print("holding %s, up-card %s" % (r.hand_name(my_hand), r.card_name(up_card)))
print("this layout -> order %+d / pass %+d" % (ordered, passed))

holding JS AS 9H 9D TC, up-card 9S
this layout -> order -2 / pass -1


In [ ]:
# 5. Expected value over many layouts.
#
# One layout tells you nothing -- the other nineteen cards happened to fall a
# particular way. Deal them again a few hundred times and solve the whole
# auction each time.
#
# A full auction is about 36 trick-play solves, so roughly 13 ms per deal.
# Raise N for a tighter interval; sampling error is the only error here, since
# each solve is exact.
N = 400
rng = random.Random(2024)

b.solve_bidding(game.deal_around(known_hand=my_hand, seat=my_seat,
                                 up_card=up_card, rng=random.Random(0),
                                 dealer=3))          # warm the JIT

values, callers, trumps = [], [], []
t0 = time.perf_counter()
for _ in range(N):
    d = game.deal_around(known_hand=my_hand, seat=my_seat, up_card=up_card,
                         rng=rng, dealer=3)
    o = b.solve_bidding(d)
    values.append(b.value_to(my_seat, o.value))      # net points to MY team
    callers.append(None if o.passed_out else o.contract.caller)
    trumps.append(None if o.passed_out else o.contract.trump)
elapsed = time.perf_counter() - t0

values = np.array(values)
se = values.std(ddof=1) / np.sqrt(N)

print("holding %s as seat %d (eldest), up-card %s"
      % (r.hand_name(my_hand), my_seat, r.card_name(up_card)))
print("%d auctions in %.1fs (%.0f ms each)\n" % (N, elapsed, 1000 * elapsed / N))
print("expected net points to my team: %+.3f   95%% CI [%+.3f, %+.3f]"
      % (values.mean(), values.mean() - 1.96 * se, values.mean() + 1.96 * se))

print("\noutcome distribution:")
for v, n in sorted(collections.Counter(values.tolist()).items()):
    print("  %+d : %5.1f%%" % (v, 100 * n / N))

print("\nwho ends up calling:")
for seat, n in sorted(collections.Counter(callers).items(), key=lambda kv: -kv[1]):
    label = ("passed out" if seat is None else
             "seat %d (%s)" % (seat, "my team" if seat % 2 == my_seat % 2
                               else "opponents"))
    print("  %-22s %5.1f%%" % (label, 100 * n / N))

print("\ntrump called:")
for suit, n in sorted(collections.Counter(trumps).items(), key=lambda kv: -kv[1]):
    print("  %-10s %5.1f%%" % ("none" if suit is None else r.suit_name(suit),
                               100 * n / N))

# What this number is, and what it isn't.
#
# Every seat above plays and bids with all four hands visible. That is exact,
# and it is not what happens at a table. The distortion is not random:
# double-dummy defenders never mis-guess who holds the left bower and never
# trump in at the wrong moment, so the model is roughly fair to hands that win
# on raw power and pessimistic about hands that win by inducing a mistake --
# exactly the marginal zone where the calling decision is actually hard.
#
# Bidding looks worse affected than play. Solved this way the auction almost
# never passes out (0 of 1600 measured), because somebody can nearly always
# find a call that is at worst harmless to their own team. Real tables throw
# hands in constantly.
#
# So treat this as the baseline: a correct answer to a question adjacent to the
# one you care about. Heuristic and sampling-based players close the gap, and
# they plug into the same machinery -- the decision rule changes, the deal, the
# auction and the scoring do not.
#
# Not yet modelled: loners, and any conditioning of the unseen cards on what
# the bidding revealed.

holding JS AS 9H 9D TC as seat 0 (eldest), up-card 9S
400 auctions in 5.0s (13 ms each)

expected net points to my team: +0.400   95% CI [+0.274, +0.526]

outcome distribution:
  -2 :   4.8%
  -1 :  34.0%
  +1 :  39.0%
  +2 :  22.2%

who ends up calling:
  seat 0 (my team)        43.2%
  seat 3 (opponents)      38.8%
  seat 2 (my team)        18.0%

trump called:
  clubs       41.2%
  spades      40.2%
  diamonds    11.2%
  hearts       7.2%
